# **Rolling Stock ETL**

### Data Fetching

In [1]:
import pandas as pd
import psycopg2

def fetch_table_to_dataframe(host_ip, database_name, user, password, table_name, port=5432):
    """
    Connects to PostgreSQL and loads the given table into a Pandas DataFrame.
    """
    try:
        # Connect to PostgreSQL
        connection = psycopg2.connect(
            host=host_ip,
            database=database_name,
            user=user,
            password=password,
            port=port
        )
        print(f"Connected successfully to {database_name} on {host_ip}")

        # Create query
        query = f"SELECT * FROM {table_name};"

        # Load into pandas DataFrame
        df = pd.read_sql_query(query, connection)
        print(f"✅ Fetched {len(df)} rows from '{table_name}'")

        return df

    except Exception as e:
        print(f"❌ Error: {e}")
        return None

    finally:
        if connection:
            connection.close()

HOST_IP = "100.95.110.69"
DATABASE_NAME = "pradigma-extractor"
USER = "postgres"
PASSWORD = "password"
PORT = 5432
TABLE_NAME = "extraction"

df_original = fetch_table_to_dataframe(HOST_IP, DATABASE_NAME, USER, PASSWORD, TABLE_NAME, PORT)

Connected successfully to pradigma-extractor on 100.95.110.69


C:\Users\win 11\AppData\Local\Temp\ipykernel_9904\2486266187.py:23: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, connection)


✅ Fetched 6834 rows from 'extraction'


In [2]:
df = df_original.copy(deep=True)
df = df[(df['status_id'] == 1) & (df['dept_name'] == 'Rolling-Stock')][['filename', 'workorder_id', 'json_data']]
# df = df[(df['dept_name'] == 'Rolling-Stock')][['filename', 'workorder_id', 'json_data']]

df.head(3)

,filename,workorder_id,json_data
0,RS_PM_WEK_4000586856.pdf,4.000587e+09,{'notification': {'notification_no': '12244350...
1,RS_PM_MTH_4000464193.pdf,4.000464e+09,{'notification': {'notification_no': '11945789...
3,RS_PM_MTH_4000446287.pdf,4.000446e+09,{'notification': {'notification_no': '11898977...


In [3]:
import pandas as pd

valid_json = df['json_data']
valid_json = valid_json[valid_json.apply(lambda x: isinstance(x, dict))]

all_keys = set()
for item in valid_json:
    all_keys.update(item.keys())

print(sorted(all_keys))


['air_standup', 'airbag-pressure', 'airbag_pressure', 'approval', 'cardan_shaft', 'cceb', 'greasing_cardan_shaft', 'notification', 'stamping', 'technician', 'train_startup_test', 'tyre-pressure', 'tyre-wear', 'tyre_pressure', 'tyre_wear', 'water_ponding', 'work_order']


### Train Start Up Test

In [4]:
df['train_startup_test'] = df['json_data'].apply(
    lambda x: x.get('train_startup_test') if isinstance(x, dict) else None
)

df['workorder_id'] = (
    df['workorder_id']
    .apply(lambda x: int(x) if pd.notnull(x) else None)
)

df[['filename', 'workorder_id', 'train_startup_test']].head(1).to_dict(orient='records')

[{'filename': 'RS_PM_WEK_4000586856.pdf',
  'workorder_id': 4000586856,
  'train_startup_test': {'4_car_train_no': '29',
   'date': '25/02/2024',
   'odometer': '147854.81',
   'train_startup_checks': {'a': {'description': 'All tractions available',
     'eca1_1': True,
     'eca1_2': 'not required',
     'ica2_1': True,
     'ica2_2': True,
     'ica3_1': True,
     'ica3_2': True,
     'eca4_1': 'not required',
     'eca4_2': True},
    'b': {'description': 'ETCS Signaling Systems available',
     'eca1': 'true',
     'ica2': 'not required',
     'ica3': 'not required',
     'eca4': 'true'},
    'c': {'description': 'EB/SB/HB and PB status shown functional normal on HMI',
     'eca1': 'true',
     'ica2': 'true',
     'ica3': 'true',
     'eca4': 'ntrueull'},
    'd': {'description': 'Check all temperature alarms normal at HMI',
     'eca1': True,
     'ica2': True,
     'ica3': True,
     'eca4': True},
    'e': {'description': 'Door open/close status shown normal',
     'eca1': Tru

In [5]:
import pandas as pd
import numpy as np

na_like_values = ['NA', 'N/A', 'NULL', 'NONE', 'NAN']

def is_na_like(val):
    if isinstance(val, (list, dict, np.ndarray)):
        return False
    try:
        if pd.isna(val):
            return True
    except Exception:
        pass
    val_str = str(val).strip().upper()
    return val_str in na_like_values

def find_na_keys(d):
    if not isinstance(d, dict):
        return []
    return [k for k, v in d.items() if is_na_like(v)]

df['na_keys'] = df['train_startup_test'].apply(find_na_keys)

df_with_na = df[df['na_keys'].apply(lambda x: len(x) > 0)]

df_with_na[['filename', 'workorder_id', 'na_keys']].head(5)

,filename,workorder_id,na_keys
22,RS_PM_MTH_4000445410.pdf,4000445410,[odometer]
53,RS_PM_MTH_4000470659.pdf,4000470659,"[4_car_train_no, date, odometer]"
176,RS_PM_QTR_4000445153.pdf,4000445153,"[4_car_train_no, odometer]"
212,RS_PM_WEK_4000445154.pdf,4000445154,"[4_car_train_no, date, odometer]"
214,RS_PM_WEK_4000444459.pdf,4000444459,"[4_car_train_no, date, odometer]"


In [6]:
from collections import Counter

na_counter = Counter(k for keys in df['na_keys'] for k in keys)
na_summary = pd.DataFrame(na_counter.items(), columns=['key', 'na_count']).sort_values('na_count', ascending=False)

print(na_summary)

              key  na_count
0        odometer        31
1  4_car_train_no        21
2            date        21


In [7]:
import numpy as np
import re
import pandas as pd

pattern = re.compile(r'^\s*(NA/NULL|NaN)\s*$', re.IGNORECASE)

def clean_value(val):
    """Clean individual values (string, dict, etc.)."""
    if isinstance(val, str):
        return '' if pattern.match(val) else val
    elif isinstance(val, dict):
        return {k: clean_value(v) for k, v in val.items()}
    elif isinstance(val, list):
        return [clean_value(v) for v in val]
    else:
        return '' if pd.isna(val) else val

df['train_startup_test'] = df['train_startup_test'].apply(clean_value)

df['train_startup_test'] = df['train_startup_test'].replace(np.nan, '', regex=True)

df['train_startup_test'].head(3)

0    {'4_car_train_no': '29', 'date': '25/02/2024',...
1    {'4_car_train_no': '25', 'date': '12/05/2022',...
3    {'4_car_train_no': '07', 'date': '25/01/2022',...
Name: train_startup_test, dtype: object

In [8]:
def flatten_with_descriptions(row):
    flat = {}

    def recurse(subdict, parent=''):
        for k, v in subdict.items():
            # if len(k) == 1 and k.isalpha():
            #     new_parent = parent
            # else:
            new_parent = f"{parent}.{k}" if parent else k

            if isinstance(v, dict):
                desc = v.get('description')
                if desc:
                    desc_key = (
                        desc.lower()
                        .replace(' ', '_')
                        .replace('/', '_')
                        .replace('&', 'and')
                    )
                    for sub_k, sub_v in v.items():
                        if sub_k != 'description':
                            flat[f"{new_parent}.{desc_key}.{sub_k}"] = sub_v
                else:
                    recurse(v, new_parent)
            else:
                flat[new_parent] = v

    recurse(row)
    return flat

flattened_rows = [flatten_with_descriptions(r) for r in df['train_startup_test']]

train_startup_test_df = pd.DataFrame(flattened_rows)

train_startup_test_df.index = df.index

train_startup_test_df['workorder_id'] = df['workorder_id'].astype('Int64')
train_startup_test_df['filename'] = df['filename']

#### Train Startup Checks - Column Analysis

In [9]:
original_cols = [
    c for c in train_startup_test_df.columns
    if c.startswith("train_startup_checks")
]

clean_map = {}

for col in original_cols:
    cleaned = col.replace("train_startup_checks.a.", "train_startup_checks.")
    cleaned = cleaned.replace("train_startup_checks.b.", "train_startup_checks.")
    clean_map[col] = cleaned


base_keys = set(clean_map.values())

workorders_per_key = {}

for key in sorted(base_keys):
    key_cols = [orig for orig, cleaned in clean_map.items() if cleaned == key]
    
    workorders = [
        int(w)
        for w in train_startup_test_df.loc[
            train_startup_test_df[key_cols].notna().any(axis=1),
            "workorder_id"
        ].dropna().unique()
    ]
    
    workorders_per_key[key] = workorders
        
for key, workorders in workorders_per_key.items():
    print(f"Key: {key} | Count: {len(workorders)}")
    # print(f"Workorder IDs with data: {workorders}")

#### Merging Columns #####

df_cleaned = train_startup_test_df.copy()

for key in base_keys:
    key_cols = [orig for orig, cleaned in clean_map.items() if cleaned == key]

    merged_column = df_cleaned[key_cols[0]].copy()

    for col_to_merge in key_cols[1:]:
        merged_column = merged_column.combine_first(df_cleaned[col_to_merge])

    df_cleaned[key] = merged_column

df_cleaned.drop(columns=original_cols, inplace=True)
train_startup_test_df = df_cleaned

Key: train_startup_checks.all_tractions_available.eca1_1 | Count: 1198
Key: train_startup_checks.all_tractions_available.eca1_2 | Count: 1198
Key: train_startup_checks.all_tractions_available.eca4_1 | Count: 1198
Key: train_startup_checks.all_tractions_available.eca4_2 | Count: 1198
Key: train_startup_checks.all_tractions_available.ica2_1 | Count: 1198
Key: train_startup_checks.all_tractions_available.ica2_2 | Count: 1198
Key: train_startup_checks.all_tractions_available.ica3_1 | Count: 1198
Key: train_startup_checks.all_tractions_available.ica3_2 | Count: 1198
Key: train_startup_checks.c.eb_sb_hb_and_pb_status_shown_functional_normal_on_hmi.eca1 | Count: 1198
Key: train_startup_checks.c.eb_sb_hb_and_pb_status_shown_functional_normal_on_hmi.eca4 | Count: 1198
Key: train_startup_checks.c.eb_sb_hb_and_pb_status_shown_functional_normal_on_hmi.ica2 | Count: 1198
Key: train_startup_checks.c.eb_sb_hb_and_pb_status_shown_functional_normal_on_hmi.ica3 | Count: 1198
Key: train_startup_checks.d.

#### Main Air Compressor - Column Analysis

In [10]:
prefix = "main_air_compressor"
train_startup_cols = [
    c for c in train_startup_test_df.columns
    if c.startswith(prefix)
]

base_keys = set(train_startup_cols)

def get_workorders_for_columns(df, key, all_cols):
    """Return all workorder_id where at least one column for this key has data."""
    key_cols = [c for c in all_cols if c.startswith(key)]

    workorders = (
        df.loc[df[key_cols].notna().any(axis=1), "workorder_id"]
        .dropna()
        .astype(int)
        .unique()
        .tolist()
    )
    return workorders

workorders_per_key = {
    key: get_workorders_for_columns(train_startup_test_df, key, train_startup_cols)
    for key in sorted(base_keys)
}

column_mapping = {
    "main_air_compressor.selected_active.eca1": [
        "main_air_compressor.a.selected_active.eca1",
        "main_air_compressor.a.selected_active_(v).eca1"
    ],
    "main_air_compressor.selected_active.ica2": [
        "main_air_compressor.a.selected_active.eca2",
        "main_air_compressor.a.selected_active_(v).eca2",
        "main_air_compressor.a.selected_active_(v).ica2"
    ],
    "main_air_compressor.selected_active.ica3": [
        "main_air_compressor.a.selected_active.eca3",
        "main_air_compressor.a.selected_active_(v).eca3",
        "main_air_compressor.a.selected_active_(v).ica3"
    ],
    "main_air_compressor.selected_active.eca4": [
        "main_air_compressor.a.selected_active.eca4",
        "main_air_compressor.a.selected_active_(v).eca4"
    ],
}

for canonical, variants in column_mapping.items():
    existing_cols = [v for v in variants if v in train_startup_test_df.columns]

    if not existing_cols:
        continue

    train_startup_test_df[canonical] = (
        train_startup_test_df[existing_cols]
            .bfill(axis=1)
            .iloc[:, 0]
            .infer_objects(copy=False)
    )

    cols_to_drop = [c for c in existing_cols if c != canonical]
    train_startup_test_df.drop(columns=cols_to_drop, inplace=True)

remaining_cols = [
    c for c in train_startup_test_df.columns
    if c.startswith(prefix)
]

general_clean_map = {}
cols_to_process = []

for col in remaining_cols:
    cleaned = col.replace(f"{prefix}.a.", f"{prefix}.")
    cleaned = cleaned.replace(f"{prefix}.b.", f"{prefix}.")
    cleaned = cleaned.replace("_(v)", "")

    if cleaned != col:
        general_clean_map[col] = cleaned
        cols_to_process.append(col)

general_base_keys = set(general_clean_map.values())

for key in sorted(general_base_keys):
    key_cols_to_merge = [
        orig for orig, cleaned in general_clean_map.items() 
        if cleaned == key
    ]
    
    existing_cols = [c for c in key_cols_to_merge if c in train_startup_test_df.columns]

    if not existing_cols:
        continue

    train_startup_test_df[key] = (
        train_startup_test_df[existing_cols]
        .bfill(axis=1)
        .iloc[:, 0]
        .infer_objects(copy=False)
    )

    train_startup_test_df.drop(columns=existing_cols, inplace=True)

# target_cols = [c for c in train_startup_test_df.columns if c.startswith(prefix)]

# print("\n--- Final Cleaned Column Counts ---")
# for i, col in enumerate(sorted(target_cols), start=1):
#     print(f"{i:3d}. {col} | Count = {train_startup_test_df[col].notna().sum()}")

#### Load Tyre Pressure

In [11]:
import pandas as pd

# --- 1. Select load_tyre_pressure columns ---
load_cols = [c for c in train_startup_test_df.columns if c.startswith("load_tyre_pressure")]

# --- 2. Create mapping from old column names to cleaned names ---
key_map = {}
for col in load_cols:
    if ".a." in col:
        parts = col.split(".a.")
        prefix = parts[0]
        last_suffix = parts[-1].split(".")[-1]
        cleaned_key = f"{prefix}.bogie1.{last_suffix}"
    elif ".b." in col:
        parts = col.split(".b.")
        prefix = parts[0]
        last_suffix = parts[-1].split(".")[-1]
        cleaned_key = f"{prefix}.bogie2.{last_suffix}"
    else:
        cleaned_key = col
    key_map[col] = cleaned_key

# --- 3. Create a new DataFrame to store merged columns ---
for cleaned_key in set(key_map.values()):
    # Get all original columns that map to this cleaned key
    orig_cols = [orig for orig, new in key_map.items() if new == cleaned_key]
    
    # Merge all columns into one: if multiple columns exist, take the first non-null value
    train_startup_test_df[cleaned_key] = train_startup_test_df[orig_cols].bfill(axis=1).iloc[:, 0]

# --- 4. Drop old original columns if desired ---
train_startup_test_df.drop(columns=load_cols, inplace=True)

# --- 5. Optional: check workorders per cleaned key ---
workorders_per_key = {}
for cleaned_key in sorted(set(key_map.values())):
    workorders = (
        train_startup_test_df.loc[train_startup_test_df[cleaned_key].notna(), "workorder_id"]
        .dropna()
        .astype(int)
        .unique()
        .tolist()
    )
    workorders_per_key[cleaned_key] = workorders

# --- 6. Print results ---
for key, workorders in workorders_per_key.items():
    print(f"Key: {key} | Count: {len(workorders)}")

Key: load_tyre_pressure_(bar)_per_bogie_facing_eca1.bogie1.eca1 | Count: 1198
Key: load_tyre_pressure_(bar)_per_bogie_facing_eca1.bogie1.eca4 | Count: 1198
Key: load_tyre_pressure_(bar)_per_bogie_facing_eca1.bogie1.ica2 | Count: 1198
Key: load_tyre_pressure_(bar)_per_bogie_facing_eca1.bogie1.ica3 | Count: 1198
Key: load_tyre_pressure_(bar)_per_bogie_facing_eca1.bogie2.eca1 | Count: 1198
Key: load_tyre_pressure_(bar)_per_bogie_facing_eca1.bogie2.eca4 | Count: 1198
Key: load_tyre_pressure_(bar)_per_bogie_facing_eca1.bogie2.ica2 | Count: 1198
Key: load_tyre_pressure_(bar)_per_bogie_facing_eca1.bogie2.ica3 | Count: 1198


#### Passenger Door Functionality Check

In [12]:
passenger_cols = [c for c in train_startup_test_df.columns if c.startswith("passenger_door_functionality_check")]

# --- 2. Create mapping from old column names to cleaned names ---
key_map = {}

for col in passenger_cols:
    if ".a." in col:
        parts = col.split(".a.")
        prefix = parts[0]
        last_suffix = parts[-1].split(".")[-1]
        cleaned_key = f"{prefix}.passenger_door1.{last_suffix}"
    elif ".b." in col:
        parts = col.split(".b.")
        prefix = parts[0]
        last_suffix = parts[-1].split(".")[-1]
        cleaned_key = f"{prefix}.passenger_door2.{last_suffix}"
    elif ".c." in col:
        parts = col.split(".c.")
        prefix = parts[0]
        last_suffix = parts[-1].split(".")[-1]
        cleaned_key = f"{prefix}.passenger_door3.{last_suffix}"
    elif ".d." in col:
        parts = col.split(".d.")
        prefix = parts[0]
        last_suffix = parts[-1].split(".")[-1]
        cleaned_key = f"{prefix}.passenger_door4.{last_suffix}"
    else:
        continue  # skip columns that don't match

    key_map[col] = cleaned_key

# --- 3. Merge original columns into cleaned columns ---
for cleaned_key in set(key_map.values()):
    orig_cols = [orig for orig, new in key_map.items() if new == cleaned_key]
    
    # Merge values: take first non-null value across original columns
    train_startup_test_df[cleaned_key] = train_startup_test_df[orig_cols].bfill(axis=1).iloc[:, 0]

# --- 4. Drop old original columns if desired ---
train_startup_test_df.drop(columns=passenger_cols, inplace=True)

# --- 5. Optional: calculate workorders per cleaned key ---
workorders_per_key = {}
for cleaned_key in sorted(set(key_map.values())):
    workorders = (
        train_startup_test_df.loc[train_startup_test_df[cleaned_key].notna(), "workorder_id"]
        .dropna()
        .astype(int)
        .unique()
        .tolist()
    )
    workorders_per_key[cleaned_key] = workorders

# --- 6. Print results ---
for key, workorders in workorders_per_key.items():
    print(f"Key: {key} | Count: {len(workorders)}")


Key: passenger_door_functionality_check.passenger_door1.eca1 | Count: 1198
Key: passenger_door_functionality_check.passenger_door1.eca4 | Count: 1198
Key: passenger_door_functionality_check.passenger_door1.ica2 | Count: 1198
Key: passenger_door_functionality_check.passenger_door1.ica3 | Count: 1198
Key: passenger_door_functionality_check.passenger_door2.eca1 | Count: 1198
Key: passenger_door_functionality_check.passenger_door2.eca4 | Count: 1198
Key: passenger_door_functionality_check.passenger_door2.ica2 | Count: 1198
Key: passenger_door_functionality_check.passenger_door2.ica3 | Count: 1198
Key: passenger_door_functionality_check.passenger_door3.eca1 | Count: 1198
Key: passenger_door_functionality_check.passenger_door3.eca4 | Count: 1198
Key: passenger_door_functionality_check.passenger_door3.ica2 | Count: 1198
Key: passenger_door_functionality_check.passenger_door3.ica3 | Count: 1198
Key: passenger_door_functionality_check.passenger_door4.eca1 | Count: 1198
Key: passenger_door_funct

#### Traction System

In [13]:
import pandas as pd

# --- 1. Merge columns based on explicit column_mapping ---
column_mapping = {
    "traction_system.b.traction_inverter_bogie_1_health_status.eca1": [
        "traction_system.b.traction_inverter_bogie_1_health_status.eca1",
        "traction_system.b.traction_inverter_bogie_1_health_status.ica_1"
    ],
    "traction_system.c.traction_inverter_bogie_2_health_status.eca4": [
        "traction_system.c.traction_inverter_bogie_2_health_status.eca4",
        "traction_system.c.traction_inverter_bogie_2_health_status.ica_4"
    ],
}

for canonical, variants in column_mapping.items():
    existing_cols = [v for v in variants if v in train_startup_test_df.columns]
    if not existing_cols:
        continue
    
    # Merge values: take first non-null value
    train_startup_test_df[canonical] = (
        train_startup_test_df[existing_cols]
        .bfill(axis=1)
        .iloc[:, 0]
        .infer_objects(copy=False)
    )
    
    # Drop redundant columns
    cols_to_drop = [c for c in existing_cols if c != canonical]
    train_startup_test_df.drop(columns=cols_to_drop, inplace=True)

# --- 2. Identify remaining traction columns ---
traction_cols = [c for c in train_startup_test_df.columns if c.startswith("traction_system")]

# --- 3. Map to cleaned keys ---
key_map = {}

for col in traction_cols:
    if ".a." in col:
        prefix = col.split(".a.")[0]
        last_suffix = col.split(".")[-1]
        cleaned_key = f"{prefix}.coolant_level.{last_suffix}"
    elif ".b." in col:
        prefix = col.split(".b.")[0]
        last_suffix = col.split(".")[-1]
        cleaned_key = f"{prefix}.traction_inverter_bogie_1_health_status.{last_suffix}"
    elif ".c." in col:
        prefix = col.split(".c.")[0]
        last_suffix = col.split(".")[-1]
        cleaned_key = f"{prefix}.traction_inverter_bogie_2_health_status.{last_suffix}"
    else:
        continue

    key_map[col] = cleaned_key

# --- 4. Merge original columns into cleaned columns ---
for cleaned_key in set(key_map.values()):
    orig_cols = [orig for orig, new in key_map.items() if new == cleaned_key]
    
    # Merge values: take first non-null value across original columns
    train_startup_test_df[cleaned_key] = train_startup_test_df[orig_cols].bfill(axis=1).iloc[:, 0]

# --- 5. Drop old original columns ---
train_startup_test_df.drop(columns=traction_cols, inplace=True)

# --- 6. Calculate workorders per cleaned key ---
workorders_per_key = {}
for cleaned_key in sorted(set(key_map.values())):
    workorders = (
        train_startup_test_df.loc[train_startup_test_df[cleaned_key].notna(), "workorder_id"]
        .dropna()
        .astype(int)
        .unique()
        .tolist()
    )
    workorders_per_key[cleaned_key] = workorders

# --- 7. Print results ---
for key, workorders in workorders_per_key.items():
    print(f"Key: {key} | Count: {len(workorders)}")

Key: traction_system.coolant_level.eca1 | Count: 1198
Key: traction_system.coolant_level.eca4 | Count: 1198
Key: traction_system.coolant_level.ica2 | Count: 1198
Key: traction_system.coolant_level.ica3 | Count: 1198
Key: traction_system.traction_inverter_bogie_1_health_status.eca1 | Count: 1198
Key: traction_system.traction_inverter_bogie_1_health_status.eca4 | Count: 1198
Key: traction_system.traction_inverter_bogie_1_health_status.ica2 | Count: 1198
Key: traction_system.traction_inverter_bogie_1_health_status.ica3 | Count: 1198
Key: traction_system.traction_inverter_bogie_2_health_status.eca1 | Count: 1198
Key: traction_system.traction_inverter_bogie_2_health_status.eca4 | Count: 1198
Key: traction_system.traction_inverter_bogie_2_health_status.ica2 | Count: 1198
Key: traction_system.traction_inverter_bogie_2_health_status.ica3 | Count: 1198


C:\Users\win 11\AppData\Local\Temp\ipykernel_9904\2930097510.py:23: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .bfill(axis=1)


#### Apron Door

In [14]:
import pandas as pd

# --- 1. Select apron door columns ---
apron_cols = [c for c in train_startup_test_df.columns if c.startswith("apron_door_status")]

# --- 2. Create mapping from old column names to cleaned names ---
key_map = {}

for col in apron_cols:
    if ".a." in col:
        prefix = col.split(".a.")[0]
        last_suffix = col.split(".")[-1]
        cleaned_key = f"{prefix}.apron_door1.{last_suffix}"
    elif ".b." in col:
        prefix = col.split(".b.")[0]
        last_suffix = col.split(".")[-1]
        cleaned_key = f"{prefix}.apron_door2.{last_suffix}"
    elif ".c." in col:
        prefix = col.split(".c.")[0]
        last_suffix = col.split(".")[-1]
        cleaned_key = f"{prefix}.apron_door3.{last_suffix}"
    elif ".d." in col:
        prefix = col.split(".d.")[0]
        last_suffix = col.split(".")[-1]
        cleaned_key = f"{prefix}.apron_door4.{last_suffix}"
    elif ".e." in col:
        prefix = col.split(".e.")[0]
        last_suffix = col.split(".")[-1]
        cleaned_key = f"{prefix}.apron_door5.{last_suffix}"
    elif ".f." in col:
        prefix = col.split(".f.")[0]
        last_suffix = col.split(".")[-1]
        cleaned_key = f"{prefix}.apron_door6.{last_suffix}"
    else:
        continue

    key_map[col] = cleaned_key

# --- 3. Merge original columns into cleaned columns ---
for cleaned_key in set(key_map.values()):
    orig_cols = [orig for orig, new in key_map.items() if new == cleaned_key]
    
    # Merge values: take first non-null value across original columns
    train_startup_test_df[cleaned_key] = train_startup_test_df[orig_cols].bfill(axis=1).iloc[:, 0]

# --- 4. Drop old original columns ---
train_startup_test_df.drop(columns=apron_cols, inplace=True)

# --- 5. Calculate workorders per cleaned key ---
workorders_per_key = {}
for cleaned_key in sorted(set(key_map.values())):
    workorders = (
        train_startup_test_df.loc[train_startup_test_df[cleaned_key].notna(), "workorder_id"]
        .dropna()
        .astype(int)
        .unique()
        .tolist()
    )
    workorders_per_key[cleaned_key] = workorders

# --- 6. Print results ---
for key, workorders in workorders_per_key.items():
    print(f"Key: {key} | Count: {len(workorders)}")
    # Uncomment below if you want to see actual workorder IDs
    # print(f"Workorder IDs with data: {list(workorders)}")

C:\Users\win 11\AppData\Local\Temp\ipykernel_9904\3063196233.py:44: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  train_startup_test_df[cleaned_key] = train_startup_test_df[orig_cols].bfill(axis=1).iloc[:, 0]
C:\Users\win 11\AppData\Local\Temp\ipykernel_9904\3063196233.py:44: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  train_startup_test_df[cleaned_key] = train_startup_test_df[orig_cols].bfill(axis=1).iloc[:, 0]
C:\Users\win 11\AppData\Local\Temp\ipykernel_9904\3063196233.py:44: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill 

Key: apron_door_status.apron_door1.eca1 | Count: 1198
Key: apron_door_status.apron_door1.eca4 | Count: 1198
Key: apron_door_status.apron_door1.ica2 | Count: 1198
Key: apron_door_status.apron_door1.ica3 | Count: 1198
Key: apron_door_status.apron_door2.eca1 | Count: 1198
Key: apron_door_status.apron_door2.eca4 | Count: 1198
Key: apron_door_status.apron_door2.ica2 | Count: 1198
Key: apron_door_status.apron_door2.ica3 | Count: 1198
Key: apron_door_status.apron_door3.eca1 | Count: 1198
Key: apron_door_status.apron_door3.eca4 | Count: 1198
Key: apron_door_status.apron_door3.ica2 | Count: 1198
Key: apron_door_status.apron_door3.ica3 | Count: 1198
Key: apron_door_status.apron_door4.eca1 | Count: 1198
Key: apron_door_status.apron_door4.eca4 | Count: 1198
Key: apron_door_status.apron_door4.ica2 | Count: 1198
Key: apron_door_status.apron_door4.ica3 | Count: 1198
Key: apron_door_status.apron_door5.eca1 | Count: 1198
Key: apron_door_status.apron_door5.eca4 | Count: 1198
Key: apron_door_status.apron

C:\Users\win 11\AppData\Local\Temp\ipykernel_9904\3063196233.py:44: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  train_startup_test_df[cleaned_key] = train_startup_test_df[orig_cols].bfill(axis=1).iloc[:, 0]
C:\Users\win 11\AppData\Local\Temp\ipykernel_9904\3063196233.py:44: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  train_startup_test_df[cleaned_key] = train_startup_test_df[orig_cols].bfill(axis=1).iloc[:, 0]
C:\Users\win 11\AppData\Local\Temp\ipykernel_9904\3063196233.py:44: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill 

#### Brakes and Pneumatics

In [15]:
import pandas as pd

# --- 1. Replace column name substrings ---
train_startup_test_df.columns = train_startup_test_df.columns.str.replace(
    "brakes_&_pneumatic", "brakes_and_pneumatic", regex=False
)

# --- 2. Suffix mapping ---
suffix_map = {
    "eca1": "eca1",
    "eca2": "ica2",
    "ica2": "ica2",
    "ica3": "ica3",
    "eca3": "ica3",
    "eca4": "eca4",
    "ica4": "eca4",
}

# --- 3. Select brake columns ---
brake_cols = [c for c in train_startup_test_df.columns if c.startswith("brakes_and_pneumatic")]

# --- 4. Create mapping from original columns to cleaned keys ---
key_map = {}

for col in brake_cols:
    raw_suffix = col.split(".")[-1]
    last_suffix = suffix_map.get(raw_suffix, raw_suffix)

    if ".a." in col:
        prefix = col.split(".a.")[0]
        cleaned_key = f"{prefix}.self-test_ensure_correct_operation_of_sb_eb_and_wsp.{last_suffix}"
    elif ".b." in col:
        prefix = col.split(".b.")[0]
        cleaned_key = f"{prefix}.brakes_control_test_the_eb.{last_suffix}"
    else:
        continue

    key_map[col] = cleaned_key

# --- 5. Merge original columns into cleaned columns ---
for cleaned_key in set(key_map.values()):
    orig_cols = [orig for orig, new in key_map.items() if new == cleaned_key]
    
    # Merge values: take first non-null value across original columns
    train_startup_test_df[cleaned_key] = train_startup_test_df[orig_cols].bfill(axis=1).iloc[:, 0]

# --- 6. Drop old original columns ---
train_startup_test_df.drop(columns=brake_cols, inplace=True)

# --- 7. Calculate workorders per cleaned key ---
workorders_per_key = {}
for cleaned_key in sorted(set(key_map.values())):
    workorders = (
        train_startup_test_df.loc[train_startup_test_df[cleaned_key].notna(), "workorder_id"]
        .dropna()
        .astype(int)
        .unique()
        .tolist()
    )
    workorders_per_key[cleaned_key] = workorders

# --- 8. Print results ---
for key, workorders in workorders_per_key.items():
    print(f"Key: {key} | Count: {len(workorders)}")
    # Uncomment below to see the actual workorder IDs
    # print(f"Workorder IDs with data: {list(workorders)}")

Key: brakes_and_pneumatic.brakes_control_test_the_eb.eca1 | Count: 1198
Key: brakes_and_pneumatic.brakes_control_test_the_eb.eca4 | Count: 1198
Key: brakes_and_pneumatic.brakes_control_test_the_eb.ica2 | Count: 1198
Key: brakes_and_pneumatic.brakes_control_test_the_eb.ica3 | Count: 1198
Key: brakes_and_pneumatic.self-test_ensure_correct_operation_of_sb_eb_and_wsp.eca1 | Count: 1198
Key: brakes_and_pneumatic.self-test_ensure_correct_operation_of_sb_eb_and_wsp.eca4 | Count: 1198
Key: brakes_and_pneumatic.self-test_ensure_correct_operation_of_sb_eb_and_wsp.ica2 | Count: 1198
Key: brakes_and_pneumatic.self-test_ensure_correct_operation_of_sb_eb_and_wsp.ica3 | Count: 1198


C:\Users\win 11\AppData\Local\Temp\ipykernel_9904\2574077886.py:45: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  train_startup_test_df[cleaned_key] = train_startup_test_df[orig_cols].bfill(axis=1).iloc[:, 0]
C:\Users\win 11\AppData\Local\Temp\ipykernel_9904\2574077886.py:45: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  train_startup_test_df[cleaned_key] = train_startup_test_df[orig_cols].bfill(axis=1).iloc[:, 0]


#### Cab

In [16]:
import pandas as pd

# --- 1. Ensure suffix_map exists from previous brakes example ---
# If not defined yet, define it (you can adjust if cab suffixes differ)
suffix_map = {
    "eca1": "eca1",
    "eca2": "ica2",
    "ica2": "ica2",
    "ica3": "ica3",
    "eca3": "ica3",
    "eca4": "eca4",
    "ica4": "eca4",
}

# --- 2. Select cab columns ---
cab_cols = [c for c in train_startup_test_df.columns if c.startswith("cab")]

# --- 3. Create mapping from old columns to cleaned keys ---
key_map = {}

for col in cab_cols:
    raw_suffix = col.split(".")[-1]
    last_suffix = suffix_map.get(raw_suffix, raw_suffix)

    if ".a." in col:
        prefix = col.split(".a.")[0]
        cleaned_key = f"{prefix}.check_cctv_pis_and_pid.{last_suffix}"
    elif ".b." in col:
        prefix = col.split(".b.")[0]
        cleaned_key = f"{prefix}.check_fire_extinguisher_alarm_status_at_hmi.{last_suffix}"
    elif ".c." in col:
        prefix = col.split(".c.")[0]
        cleaned_key = f"{prefix}.check_pa_system.{last_suffix}"
    elif ".d." in col:
        prefix = col.split(".d.")[0]
        cleaned_key = f"{prefix}.check_wiper_operation_including_the_wiper_washer.{last_suffix}"
    elif ".e." in col:
        prefix = col.split(".e.")[0]
        cleaned_key = f"{prefix}.check_first_aid_box.{last_suffix}"
    elif ".f." in col:
        prefix = col.split(".f.")[0]
        cleaned_key = f"{prefix}.operate_headlight_and_tail_light.{last_suffix}"
    elif ".g." in col:
        prefix = col.split(".g.")[0]
        cleaned_key = f"{prefix}.check_driver_console_condition.{last_suffix}"
    elif ".h." in col:
        prefix = col.split(".h.")[0]
        cleaned_key = f"{prefix}.check_condition_of_components_at_driver's_cab.{last_suffix}"
    elif ".i." in col:
        prefix = col.split(".i.")[0]
        cleaned_key = f"{prefix}.check_seat_condition.{last_suffix}"
    elif ".j." in col:
        prefix = col.split(".j.")[0]
        cleaned_key = f"{prefix}.check_magnetic_door_functionality.{last_suffix}"
    elif ".k." in col:
        prefix = col.split(".k.")[0]
        cleaned_key = f"{prefix}.demister_functionality.{last_suffix}"
    elif ".l." in col:
        prefix = col.split(".l.")[0]
        cleaned_key = f"{prefix}.reading_light.{last_suffix}"
    elif ".m." in col:
        prefix = col.split(".m.")[0]
        cleaned_key = f"{prefix}.cabin_light.{last_suffix}"
    else:
        continue

    key_map[col] = cleaned_key

# --- 4. Merge original columns into cleaned columns ---
for cleaned_key in set(key_map.values()):
    orig_cols = [orig for orig, mapped_key in key_map.items() if mapped_key == cleaned_key]
    if not orig_cols:
        continue

    # Merge values: take first non-null value
    train_startup_test_df[cleaned_key] = train_startup_test_df[orig_cols].bfill(axis=1).iloc[:, 0]

# --- 5. Drop old original columns ---
train_startup_test_df.drop(columns=cab_cols, inplace=True)

# --- 6. Calculate workorders per cleaned key ---
workorders_per_key = {}
for cleaned_key in sorted(set(key_map.values())):
    workorders = (
        train_startup_test_df.loc[train_startup_test_df[cleaned_key].notna(), "workorder_id"]
        .dropna()
        .astype(int)
        .unique()
        .tolist()
    )
    workorders_per_key[cleaned_key] = workorders

# --- 7. Print results ---
for key, workorders in workorders_per_key.items():
    print(f"Key: {key} | Count: {len(workorders)}")
    # Uncomment below to see actual workorder IDs
    # print(f"Workorder IDs with data: {list(workorders)}")


C:\Users\win 11\AppData\Local\Temp\ipykernel_9904\2735368078.py:76: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train_startup_test_df[cleaned_key] = train_startup_test_df[orig_cols].bfill(axis=1).iloc[:, 0]
C:\Users\win 11\AppData\Local\Temp\ipykernel_9904\2735368078.py:76: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train_startup_test_df[cleaned_key] = train_startup_test_df[orig_cols].bfill(axis=1).iloc[:, 0]
C:\Users\win 11\AppData\Local\Temp\ipykernel_9904\2735368078.py:76: PerformanceWarning: DataFrame is highly fragment

Key: cab.cabin_light.eca1 | Count: 1198
Key: cab.cabin_light.eca4 | Count: 1198
Key: cab.cabin_light.ica2 | Count: 1198
Key: cab.cabin_light.ica3 | Count: 1198
Key: cab.check_cctv_pis_and_pid.eca1 | Count: 1198
Key: cab.check_cctv_pis_and_pid.eca4 | Count: 1198
Key: cab.check_cctv_pis_and_pid.ica2 | Count: 1198
Key: cab.check_cctv_pis_and_pid.ica3 | Count: 1198
Key: cab.check_condition_of_components_at_driver's_cab.eca1 | Count: 1198
Key: cab.check_condition_of_components_at_driver's_cab.eca4 | Count: 1198
Key: cab.check_condition_of_components_at_driver's_cab.ica2 | Count: 1198
Key: cab.check_condition_of_components_at_driver's_cab.ica3 | Count: 1198
Key: cab.check_driver_console_condition.eca1 | Count: 1198
Key: cab.check_driver_console_condition.eca4 | Count: 1198
Key: cab.check_driver_console_condition.ica2 | Count: 1198
Key: cab.check_driver_console_condition.ica3 | Count: 1198
Key: cab.check_fire_extinguisher_alarm_status_at_hmi.eca1_1 | Count: 1198
Key: cab.check_fire_extinguish

C:\Users\win 11\AppData\Local\Temp\ipykernel_9904\2735368078.py:76: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train_startup_test_df[cleaned_key] = train_startup_test_df[orig_cols].bfill(axis=1).iloc[:, 0]
C:\Users\win 11\AppData\Local\Temp\ipykernel_9904\2735368078.py:76: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train_startup_test_df[cleaned_key] = train_startup_test_df[orig_cols].bfill(axis=1).iloc[:, 0]
C:\Users\win 11\AppData\Local\Temp\ipykernel_9904\2735368078.py:76: PerformanceWarning: DataFrame is highly fragment

#### Saloon

In [17]:
import pandas as pd

# --- 1. Ensure suffix_map exists (from previous sections) ---
# If not defined yet, define it
suffix_map = {
    "eca1": "eca1",
    "eca2": "ica2",
    "ica2": "ica2",
    "ica3": "ica3",
    "eca3": "ica3",
    "eca4": "eca4",
    "ica4": "eca4",
}

# --- 2. Select saloon columns ---
saloon_cols = [c for c in train_startup_test_df.columns if c.startswith("saloon")]

# --- 3. Create mapping from old columns to cleaned keys ---
key_map = {}

for col in saloon_cols:
    raw_suffix = col.split(".")[-1]
    last_suffix = suffix_map.get(raw_suffix, raw_suffix)

    if ".a." in col:
        prefix = col.split(".a.")[0]
        cleaned_key = f"{prefix}.check_interior_lighting_functioning_including_the_advert_light.{last_suffix}"
    elif ".b." in col:
        prefix = col.split(".b.")[0]
        cleaned_key = f"{prefix}.check_vac_functionality.{last_suffix}"
    elif ".c." in col:
        prefix = col.split(".c.")[0]
        cleaned_key = f"{prefix}.operate_passenger_door_open_close_and_ensure_all_doors_are_function_normal.{last_suffix}"
    elif ".d." in col:
        prefix = col.split(".d.")[0]
        cleaned_key = f"{prefix}.check_door_and_window_glass_and_rubber_seal_for_cracks_and_damages.{last_suffix}"
    elif ".e." in col:
        prefix = col.split(".e.")[0]
        cleaned_key = f"{prefix}.inspect_door_emergency_egress_handle_intact_and_in_correct_position.{last_suffix}"
    elif ".f." in col:
        prefix = col.split(".f.")[0]
        cleaned_key = f"{prefix}.check_ventilation_windows_in_close_and_unlock_position.{last_suffix}"
    elif ".g." in col:
        prefix = col.split(".g.")[0]
        cleaned_key = f"{prefix}.check_all_windows_glass_and_seal_for_cracks_and_damages_(ensure_glass_surface_clean).{last_suffix}"
    elif ".h." in col:
        prefix = col.split(".h.")[0]
        cleaned_key = f"{prefix}.functional_check_the_disable_seat_belts.{last_suffix}"
    elif ".i." in col:
        prefix = col.split(".i.")[0]
        cleaned_key = f"{prefix}.check_thread_plate_for_any_abnormalities_(gangway)_and_any_abnormal_condition_and_hazardous.{last_suffix}"
    elif ".j." in col:
        prefix = col.split(".j.")[0]
        cleaned_key = f"{prefix}.check_the_help_point_operation.{last_suffix}"
    else:
        continue

    key_map[col] = cleaned_key

# --- 4. Merge original columns into cleaned columns ---
for cleaned_key in set(key_map.values()):
    orig_cols = [orig for orig, mapped_key in key_map.items() if mapped_key == cleaned_key]
    if not orig_cols:
        continue

    # Merge values: take first non-null value
    train_startup_test_df[cleaned_key] = train_startup_test_df[orig_cols].bfill(axis=1).iloc[:, 0]

# --- 5. Drop old original columns ---
train_startup_test_df.drop(columns=saloon_cols, inplace=True)

# --- 6. Calculate workorders per cleaned key ---
workorders_per_key = {}
for cleaned_key in sorted(set(key_map.values())):
    workorders = (
        train_startup_test_df.loc[train_startup_test_df[cleaned_key].notna(), "workorder_id"]
        .dropna()
        .astype(int)
        .unique()
        .tolist()
    )
    workorders_per_key[cleaned_key] = workorders

# --- 7. Print results ---
for key, workorders in workorders_per_key.items():
    print(f"Key: {key} | Count: {len(workorders)}")
    # Uncomment below to see actual workorder IDs
    # print(f"Workorder IDs with data: {list(workorders)}")

C:\Users\win 11\AppData\Local\Temp\ipykernel_9904\493616409.py:67: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train_startup_test_df[cleaned_key] = train_startup_test_df[orig_cols].bfill(axis=1).iloc[:, 0]
C:\Users\win 11\AppData\Local\Temp\ipykernel_9904\493616409.py:67: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train_startup_test_df[cleaned_key] = train_startup_test_df[orig_cols].bfill(axis=1).iloc[:, 0]
C:\Users\win 11\AppData\Local\Temp\ipykernel_9904\493616409.py:67: PerformanceWarning: DataFrame is highly fragmented.

Key: saloon.check_all_windows_glass_and_seal_for_cracks_and_damages_(ensure_glass_surface_clean).eca1 | Count: 1198
Key: saloon.check_all_windows_glass_and_seal_for_cracks_and_damages_(ensure_glass_surface_clean).eca4 | Count: 1198
Key: saloon.check_all_windows_glass_and_seal_for_cracks_and_damages_(ensure_glass_surface_clean).ica2 | Count: 1198
Key: saloon.check_all_windows_glass_and_seal_for_cracks_and_damages_(ensure_glass_surface_clean).ica3 | Count: 1198
Key: saloon.check_door_and_window_glass_and_rubber_seal_for_cracks_and_damages.eca1 | Count: 1198
Key: saloon.check_door_and_window_glass_and_rubber_seal_for_cracks_and_damages.eca4 | Count: 1198
Key: saloon.check_door_and_window_glass_and_rubber_seal_for_cracks_and_damages.ica2 | Count: 1198
Key: saloon.check_door_and_window_glass_and_rubber_seal_for_cracks_and_damages.ica3 | Count: 1198
Key: saloon.check_interior_lighting_functioning_including_the_advert_light.eca1 | Count: 1198
Key: saloon.check_interior_lighting_functioning_in

#### Exterior

In [18]:
import pandas as pd

# --- 1. Ensure suffix_map exists (from previous sections) ---
# If not defined yet, define it
suffix_map = {
    "eca1": "eca1",
    "eca2": "ica2",
    "ica2": "ica2",
    "ica3": "ica3",
    "eca3": "ica3",
    "eca4": "eca4",
    "ica4": "eca4",
}

# --- 2. Select exterior columns ---
exterior_cols = [c for c in train_startup_test_df.columns if c.startswith("exterior")]

# --- 3. Create mapping from old columns to cleaned keys ---
key_map = {}

for col in exterior_cols:
    raw_suffix = col.split(".")[-1]
    last_suffix = suffix_map.get(raw_suffix, raw_suffix)

    if ".a." in col:
        prefix = col.split(".a.")[0]
        cleaned_key = f"{prefix}.check_apron_door_locked_and_secured_and_ensure_no_alarm_at_hmi_(when_train_stabling_in_depot).{last_suffix}"
    elif ".b." in col:
        prefix = col.split(".b.")[0]
        cleaned_key = f"{prefix}.check_headlight_and_tail_light_for_crack_and_damages.{last_suffix}"
    elif ".c." in col:
        prefix = col.split(".c.")[0]
        cleaned_key = f"{prefix}.amber_light_indicator_status.{last_suffix}"
    else:
        continue

    key_map[col] = cleaned_key

# --- 4. Merge original columns into cleaned columns ---
for cleaned_key in set(key_map.values()):
    orig_cols = [orig for orig, mapped_key in key_map.items() if mapped_key == cleaned_key]
    if not orig_cols:
        continue

    # Merge values: take first non-null value
    train_startup_test_df[cleaned_key] = train_startup_test_df[orig_cols].bfill(axis=1).iloc[:, 0]

# --- 5. Drop old original columns ---
train_startup_test_df.drop(columns=exterior_cols, inplace=True)

# --- 6. Calculate workorders per cleaned key ---
workorders_per_key = {}
for cleaned_key in sorted(set(key_map.values())):
    workorders = (
        train_startup_test_df.loc[train_startup_test_df[cleaned_key].notna(), "workorder_id"]
        .dropna()
        .astype(int)
        .unique()
        .tolist()
    )
    workorders_per_key[cleaned_key] = workorders

# --- 7. Print results ---
for key, workorders in workorders_per_key.items():
    print(f"Key: {key} | Count: {len(workorders)}")
    # Uncomment below to see actual workorder IDs
    # print(f"Workorder IDs with data: {list(workorders)}")


C:\Users\win 11\AppData\Local\Temp\ipykernel_9904\2640910314.py:46: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train_startup_test_df[cleaned_key] = train_startup_test_df[orig_cols].bfill(axis=1).iloc[:, 0]
C:\Users\win 11\AppData\Local\Temp\ipykernel_9904\2640910314.py:46: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train_startup_test_df[cleaned_key] = train_startup_test_df[orig_cols].bfill(axis=1).iloc[:, 0]
C:\Users\win 11\AppData\Local\Temp\ipykernel_9904\2640910314.py:46: PerformanceWarning: DataFrame is highly fragment

Key: exterior.amber_light_indicator_status.eca1 | Count: 1198
Key: exterior.amber_light_indicator_status.eca4 | Count: 1198
Key: exterior.amber_light_indicator_status.ica2 | Count: 1198
Key: exterior.amber_light_indicator_status.ica3 | Count: 1198
Key: exterior.check_apron_door_locked_and_secured_and_ensure_no_alarm_at_hmi_(when_train_stabling_in_depot).eca1 | Count: 1198
Key: exterior.check_apron_door_locked_and_secured_and_ensure_no_alarm_at_hmi_(when_train_stabling_in_depot).eca4 | Count: 1198
Key: exterior.check_apron_door_locked_and_secured_and_ensure_no_alarm_at_hmi_(when_train_stabling_in_depot).ica2 | Count: 1198
Key: exterior.check_apron_door_locked_and_secured_and_ensure_no_alarm_at_hmi_(when_train_stabling_in_depot).ica3 | Count: 1198
Key: exterior.check_headlight_and_tail_light_for_crack_and_damages.eca1 | Count: 1198
Key: exterior.check_headlight_and_tail_light_for_crack_and_damages.eca4 | Count: 1198
Key: exterior.check_headlight_and_tail_light_for_crack_and_damages.ica2 

In [19]:
cols_to_update = [
    "train_startup_checks.all_tractions_available.eca1_2",
    "train_startup_checks.all_tractions_available.eca4_1",
    "train_startup_checks.etcs_signaling_systems_available.ica2",
    "train_startup_checks.etcs_signaling_systems_available.ica3",
    "main_air_compressor.selected_active.ica2",
    "main_air_compressor.selected_active.ica3",
    "traction_system.traction_inverter_bogie_1_health_status.eca1",
    "traction_system.traction_inverter_bogie_2_health_status.eca4",
    "brakes_and_pneumatic.self-test_ensure_correct_operation_of_sb_eb_and_wsp.ica2",
    "brakes_and_pneumatic.self-test_ensure_correct_operation_of_sb_eb_and_wsp.ica3",
    "cab.check_pa_system.ica2",
    "cab.check_pa_system.ica3",
    "cab.check_wiper_operation_including_the_wiper_washer.ica2",
    "cab.check_wiper_operation_including_the_wiper_washer.ica3",
    "cab.check_first_aid_box.ica2",
    "cab.check_first_aid_box.ica3",
    "cab.operate_headlight_and_tail_light.ica2",
    "cab.operate_headlight_and_tail_light.ica3",
    "cab.check_driver_console_condition.ica2",
    "cab.check_driver_console_condition.ica3",
    "cab.check_condition_of_components_at_driver's_cab.ica2",
    "cab.check_condition_of_components_at_driver's_cab.ica3",
    "cab.check_seat_condition.ica2",
    "cab.check_seat_condition.ica3",
    "cab.check_magnetic_door_functionality.ica2",
    "cab.check_magnetic_door_functionality.ica3",
    "cab.demister_functionality.ica2",
    "cab.demister_functionality.ica3",
    "cab.reading_light.ica2",
    "cab.reading_light.ica3",
    "cab.cabin_light.ica2",
    "cab.cabin_light.ica3",
    "exterior.check_headlight_and_tail_light_for_crack_and_damages.ica2",
    "exterior.check_headlight_and_tail_light_for_crack_and_damages.ica3",
]

existing_cols = [c for c in cols_to_update if c in train_startup_test_df.columns]

train_startup_test_df[existing_cols] = "not required"

print(f"✅ Updated {len(existing_cols)} columns to 'not required':")


✅ Updated 34 columns to 'not required':


In [20]:
section_order = [
    'filename',
    "4_car_train_no",
    "date",
    "odometer",
    "train_startup_checks",
    "main_air_compressor",
    "load_tyre_pressure_(bar)_per_bogie_facing_eca1",
    "passenger_door_functionality_check",
    "traction_system",
    "apron_door_status",
    "brakes_and_pneumatic",
    "cab",
    "saloon",
    "exterior",
    "checked_by"
]

ordered_cols = ['workorder_id']
for prefix in section_order:
    section_cols = [col for col in train_startup_test_df.columns if col.startswith(prefix)]
    ordered_cols.extend(section_cols)

remaining_cols = [col for col in train_startup_test_df.columns if col not in ordered_cols]
ordered_cols.extend(remaining_cols)

train_startup_test_df = train_startup_test_df[ordered_cols]

print("✅ Columns reordered successfully:")
for prefix in section_order:
    matched = [c for c in train_startup_test_df.columns if c.startswith(prefix)]
    if matched:
        print(f" - {prefix}: {len(matched)} columns")

✅ Columns reordered successfully:
 - filename: 1 columns
 - 4_car_train_no: 1 columns
 - date: 1 columns
 - odometer: 1 columns
 - train_startup_checks: 12 columns
 - main_air_compressor: 4 columns
 - load_tyre_pressure_(bar)_per_bogie_facing_eca1: 8 columns
 - passenger_door_functionality_check: 16 columns
 - traction_system: 12 columns
 - apron_door_status: 24 columns
 - brakes_and_pneumatic: 8 columns
 - cab: 54 columns
 - saloon: 40 columns
 - exterior: 12 columns
 - checked_by: 2 columns


In [ ]:
# import pandas as pd
# output_path = 'output/train_startup_test.xlsx'

# with pd.ExcelWriter(output_path, engine='openpyxl', mode='w') as writer:
#     train_startup_test_df.to_excel(writer, index=False, sheet_name='train_startup_test')

# print(f"✅ Exported successfully to '{output_path}'")

import pandas as pd
import os

output_path = '../../output/rolling_stock.xlsx'

# Ensure folder exists
os.makedirs(os.path.dirname(output_path), exist_ok=True)

# Write ONLY the train_startup_test sheet
with pd.ExcelWriter(
    output_path,
    engine='openpyxl',
    mode='a',                # append to the file
    if_sheet_exists='replace'  # replace only this sheet
) as writer:
    train_startup_test_df.to_excel(writer, index=False, sheet_name='train_startup_test')

print(f"✅ 'train_startup_test' sheet updated successfully in '{output_path}'")


✅ 'train_startup_test' sheet updated successfully in '../../output/rolling_stock.xlsx'
